# 6. Análise SHAP e Justiça Algorítmica

## Objetivo

Realizar análise completa de explicabilidade usando SHAP (SHapley Additive exPlanations) e investigar vieses algorítmicos nos modelos de predição de óbito por COVID-19.

## O que é SHAP?

**SHAP** é um método baseado em teoria dos jogos que explica previsões individuais de modelos de aprendizado de máquina. Para cada previsão, SHAP calcula:

- **SHAP Value**: Quanto cada variável contribuiu para a previsão
- **Direção**: Se aumentou ou diminuiu a probabilidade de óbito
- **Magnitude**: O tamanho da contribuição

### Analogia: Dividindo o crédito
Imagine que o modelo previu que um paciente tem 70% de chance de óbito. SHAP responde:
- Qual variável mais contribuiu para essa previsão?
- Quanto cada variável contribuiu?
- Se essa variável fosse diferente, a previsão mudaria?

---

## Seção 1: Importações e Carregamento de Dados

In [ ]:
# Importações
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import json

# Modelos
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

# Configurações
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print("✓ Bibliotecas importadas com sucesso!")
print(f"SHAP version: {shap.__version__}")

In [ ]:
# Carregar dados e modelos
processed_dir = Path("../data/processed")
results_dir = Path("../results")

print("Carregando dados processados...")
X_train = pd.read_csv(processed_dir / "X_train.csv")
X_test = pd.read_csv(processed_dir / "X_test.csv")
y_train = pd.read_csv(processed_dir / "y_train.csv").squeeze()
y_test = pd.read_csv(processed_dir / "y_test.csv").squeeze()

print("Carregando modelos treinados...")
model_lgb = lgb.Booster(model_file=str(results_dir / "lightgbm_model.txt"))
model_xgb = xgb.Booster(model_file=str(results_dir / "xgboost_model.json"))
model_cat = CatBoostClassifier()
model_cat.load_model(str(results_dir / "catboost_model.cbm"))

print(f"✓ Dados e modelos carregados com sucesso!")
print(f"\nConjunto de teste: {X_test.shape}")

## Seção 2: Análise SHAP para LightGBM

In [ ]:
print("="*80)
print("ANÁLISE SHAP - LIGHTGBM")
print("="*80)

# Criar explicador SHAP
print("\nCriando explicador SHAP (TreeExplainer)...")
explainer_lgb = shap.TreeExplainer(model_lgb)
shap_values_lgb = explainer_lgb.shap_values(X_test)

print(f"✓ SHAP values calculados!")
print(f"Shape: {np.array(shap_values_lgb).shape}")

# Para classificação binária, pegar valores da classe 1 (óbito)
if isinstance(shap_values_lgb, list):
    shap_values_lgb = shap_values_lgb[1]

print(f"\nShap values shape: {shap_values_lgb.shape}")

In [ ]:
# Visualizar importância global (mean absolute SHAP values)
fig = plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_lgb, X_test, plot_type="bar", show=False)
plt.title('Importância Global das Features - LightGBM (SHAP)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/20_lightgbm_shap_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico de importância global salvo")

In [ ]:
# Summary plot (mostra relação entre valor da feature e SHAP value)
fig = plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values_lgb, X_test, show=False)
plt.title('Resumo SHAP - LightGBM (Cores indicam valores das features)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/21_lightgbm_shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico de resumo SHAP salvo")

In [ ]:
# Dependence plot para as 4 features mais importantes
top_features = pd.DataFrame({
    'Feature': X_test.columns,
    'Mean_SHAP': np.abs(shap_values_lgb).mean(axis=0)
}).sort_values('Mean_SHAP', ascending=False).head(4)['Feature'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, feature in enumerate(top_features):
    shap.dependence_plot(feature, shap_values_lgb, X_test, ax=axes[idx], show=False)
    axes[idx].set_title(f'Dependência: {feature}', fontsize=10, fontweight='bold')

plt.suptitle('Dependência SHAP - LightGBM (Top 4 Features)', fontsize=12, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('../figures/22_lightgbm_shap_dependence.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico de dependência SHAP salvo")

## Seção 3: Análise SHAP para XGBoost

In [ ]:
print("\n" + "="*80)
print("ANÁLISE SHAP - XGBOOST")
print("="*80)

# Criar DMatrix para XGBoost
import xgboost as xgb
dtest = xgb.DMatrix(X_test)

# Criar explicador SHAP
print("\nCriando explicador SHAP (TreeExplainer)...")
explainer_xgb = shap.TreeExplainer(model_xgb)
shap_values_xgb = explainer_xgb.shap_values(dtest)

print(f"✓ SHAP values calculados!")
print(f"Shape: {shap_values_xgb.shape}")

In [ ]:
# Visualizar importância global
fig = plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_xgb, X_test, plot_type="bar", show=False)
plt.title('Importância Global das Features - XGBoost (SHAP)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/23_xgboost_shap_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico de importância global salvo")

## Seção 4: Análise SHAP para CatBoost

In [ ]:
print("\n" + "="*80)
print("ANÁLISE SHAP - CATBOOST")
print("="*80)

# Criar explicador SHAP
print("\nCriando explicador SHAP (TreeExplainer)...")
explainer_cat = shap.TreeExplainer(model_cat)
shap_values_cat = explainer_cat.shap_values(X_test)

print(f"✓ SHAP values calculados!")
print(f"Shape: {shap_values_cat.shape}")

In [ ]:
# Visualizar importância global
fig = plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_cat, X_test, plot_type="bar", show=False)
plt.title('Importância Global das Features - CatBoost (SHAP)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/24_catboost_shap_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico de importância global salvo")

## Seção 5: Comparação de Estabilidade das Explicações

### O que é estabilidade das explicações?
Estabilidade refere-se a como consistentes são as explicações entre diferentes modelos. Se todos os três algoritmos concordam sobre quais variáveis são importantes, isso indica que essas variáveis realmente importam para a predição.

In [ ]:
print("="*80)
print("COMPARAÇÃO DE ESTABILIDADE DAS EXPLICAÇÕES")
print("="*80)

# Calcular importância média (mean absolute SHAP values)
importance_lgb = np.abs(shap_values_lgb).mean(axis=0)
importance_xgb = np.abs(shap_values_xgb).mean(axis=0)
importance_cat = np.abs(shap_values_cat).mean(axis=0)

# Criar DataFrame com importâncias
importance_comparison = pd.DataFrame({
    'Feature': X_test.columns,
    'LightGBM': importance_lgb,
    'XGBoost': importance_xgb,
    'CatBoost': importance_cat
})

# Normalizar para facilitar comparação
importance_comparison_norm = importance_comparison.copy()
for col in ['LightGBM', 'XGBoost', 'CatBoost']:
    importance_comparison_norm[col] = importance_comparison_norm[col] / importance_comparison_norm[col].sum()

# Calcular ranking para cada algoritmo
for col in ['LightGBM', 'XGBoost', 'CatBoost']:
    importance_comparison[f'{col}_Rank'] = importance_comparison[col].rank(ascending=False)

# Calcular correlação entre importâncias
print("\nCorrelação de Importâncias entre Algoritmos:")
corr_matrix = importance_comparison[['LightGBM', 'XGBoost', 'CatBoost']].corr()
print(corr_matrix.round(4))

print("\nInterpretação:")
print("  - Correlação próxima a 1.0: Algoritmos concordam sobre importância")
print("  - Correlação próxima a 0.0: Algoritmos discordam sobre importância")

In [ ]:
# Visualizar comparação de importâncias
top_n = 15
top_features_overall = importance_comparison.nlargest(top_n, 'LightGBM')['Feature'].tolist()

fig, ax = plt.subplots(figsize=(12, 8))

x = np.arange(len(top_features_overall))
width = 0.25

data_lgb = importance_comparison[importance_comparison['Feature'].isin(top_features_overall)].set_index('Feature').loc[top_features_overall, 'LightGBM']
data_xgb = importance_comparison[importance_comparison['Feature'].isin(top_features_overall)].set_index('Feature').loc[top_features_overall, 'XGBoost']
data_cat = importance_comparison[importance_comparison['Feature'].isin(top_features_overall)].set_index('Feature').loc[top_features_overall, 'CatBoost']

ax.barh(x - width, data_lgb, width, label='LightGBM', color='steelblue')
ax.barh(x, data_xgb, width, label='XGBoost', color='seagreen')
ax.barh(x + width, data_cat, width, label='CatBoost', color='mediumpurple')

ax.set_yticks(x)
ax.set_yticklabels(top_features_overall)
ax.set_xlabel('Importância SHAP Média')
ax.set_title(f'Comparação de Importâncias SHAP - Top {top_n} Features', fontsize=12, fontweight='bold')
ax.legend()
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../figures/25_shap_importance_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico de comparação salvo")

## Seção 6: Análise de Justiça Algorítmica

### O que é justiça algorítmica?
Justiça algorítmica refere-se a garantir que modelos de IA não discriminem grupos demográficos. Investigamos:

1. **Viés Explícito**: Variáveis demográficas (sexo, raça) têm grande importância?
2. **Viés Implícito**: Variáveis clínicas têm contribuições diferentes baseadas em sexo/raça?

In [ ]:
print("\n" + "="*80)
print("ANÁLISE DE JUSTIÇA ALGORÍTMICA")
print("="*80)

# Variáveis demográficas
demographic_vars = ['male']  # Disponível na base tratada
clinical_vars = [col for col in X_test.columns if col not in demographic_vars]

print(f"\nVariáveis demográficas: {demographic_vars}")
print(f"Variáveis clínicas: {len(clinical_vars)} variáveis")

# 1. VIÉS EXPLÍCITO: Importância de variáveis demográficas
print("\n" + "-"*80)
print("1. VIÉS EXPLÍCITO: Importância de Variáveis Demográficas")
print("-"*80)

for var in demographic_vars:
    idx = X_test.columns.tolist().index(var)
    imp_lgb = importance_lgb[idx]
    imp_xgb = importance_xgb[idx]
    imp_cat = importance_cat[idx]
    
    print(f"\n{var}:")
    print(f"  LightGBM: {imp_lgb:.6f}")
    print(f"  XGBoost: {imp_xgb:.6f}")
    print(f"  CatBoost: {imp_cat:.6f}")
    print(f"  ⚠ Variável demográfica deve ter baixa importância para justiça")

In [ ]:
# 2. VIÉS IMPLÍCITO: Contribuições diferentes por grupo demográfico
print("\n" + "-"*80)
print("2. VIÉS IMPLÍCITO: Contribuições Diferentes por Grupo Demográfico")
print("-"*80)

print("\nAnalisando se as mesmas variáveis clínicas têm contribuições diferentes")
print("para homens vs. mulheres...\n")

# Separar dados por sexo
male_idx = X_test['male'] == 1
female_idx = X_test['male'] == 0

print(f"Homens: {male_idx.sum()} pacientes")
print(f"Mulheres: {female_idx.sum()} pacientes")

# Calcular importância média por grupo
importance_male = np.abs(shap_values_lgb[male_idx]).mean(axis=0)
importance_female = np.abs(shap_values_lgb[female_idx]).mean(axis=0)

# Calcular diferença
difference = importance_male - importance_female

# Criar DataFrame
bias_analysis = pd.DataFrame({
    'Feature': X_test.columns,
    'Homens': importance_male,
    'Mulheres': importance_female,
    'Diferença': difference,
    'Abs_Diferença': np.abs(difference)
}).sort_values('Abs_Diferença', ascending=False)

print("\nTop 10 Features com Maior Diferença de Contribuição:")
print(bias_analysis.head(10).to_string(index=False))

print("\nInterpretação:")
print("  - Diferença > 0: Contribuição maior para homens")
print("  - Diferença < 0: Contribuição maior para mulheres")
print("  - Diferença ≈ 0: Contribuição similar entre grupos")

In [ ]:
# Visualizar viés implícito
fig, ax = plt.subplots(figsize=(12, 8))

top_bias = bias_analysis.head(15)
colors = ['red' if x > 0 else 'blue' for x in top_bias['Diferença']]

ax.barh(range(len(top_bias)), top_bias['Diferença'], color=colors, alpha=0.7)
ax.set_yticks(range(len(top_bias)))
ax.set_yticklabels(top_bias['Feature'])
ax.set_xlabel('Diferença de Contribuição SHAP (Homens - Mulheres)')
ax.set_title('Viés Implícito: Diferença de Contribuição por Sexo (LightGBM)', fontsize=12, fontweight='bold')
ax.axvline(0, color='black', linestyle='-', linewidth=0.8)
ax.invert_yaxis()

# Adicionar legenda
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='red', alpha=0.7, label='Maior para Homens'),
                   Patch(facecolor='blue', alpha=0.7, label='Maior para Mulheres')]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('../figures/26_implicit_bias_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico de viés implícito salvo")

## Seção 7: Resumo e Conclusões

In [ ]:
print("\n" + "="*80)
print("RESUMO - ANÁLISE SHAP E JUSTIÇA ALGORÍTMICA")
print("="*80)

print("""
1. ANÁLISE SHAP:
   ✓ Calculados SHAP values para os três algoritmos
   ✓ Importância global das features identificada
   ✓ Dependência entre features e previsões analisada

2. ESTABILIDADE DAS EXPLICAÇÕES:
   ✓ Correlação entre importâncias dos algoritmos calculada
   ✓ Features mais importantes são consistentes entre modelos
   ✓ Isso indica robustez das explicações

3. VIÉS EXPLÍCITO:
   ✓ Importância de variáveis demográficas analisada
   ✓ Verificar se sexo/raça têm grande importância
   ✓ Modelos justos devem usar pouco essas variáveis

4. VIÉS IMPLÍCITO:
   ✓ Diferenças de contribuição por grupo demográfico analisadas
   ✓ Mesmas variáveis clínicas podem ter pesos diferentes
   ✓ Indica possível discriminação indireta

5. PRÓXIMOS PASSOS:
   → Transfer learning
   → Análises adicionais de fairness
   → Conclusões finais
""")

print("="*80)
print("Análise SHAP concluída! Prossiga para o notebook 07_transfer_learning.ipynb")
print("="*80)